# Stock Investment: The p-index Approach

## Paper Citation
Xinzhao Xie, Bopei Nie, Kuo-Ping Chang. "Stock Investment: The p-index Approach". Published: 2026-06-07. [ArXiv: 2606.08569](https://arxiv.org/abs/2606.08569).

## Strategy Description
This notebook implements the p-index approach for stock investment as described in the paper. The p-index measures the insurance fee for each insured dollar to guarantee that the asset achieves at least a delta rate of return on a specified future date. The strategy evaluates different investment strategies using the p-index risk measure.

The notebook follows the CRISP-TIQ 6-phase structure:
1. Configuration
2. Data Download & Feature Computation
3. Signal Generation & Portfolio Construction
4. Backtesting
5. Performance Metrics
6. Monitoring

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
RISK_FREE_RATE = 0.02
HOLDING_PERIOD = '1mo'
MOMENTUM_PERIOD = 21
CONTRARIA_PERIOD = 63


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start='2018-01-01', interval=HOLDING_PERIOD)
prices = data['Adj Close']
returns = prices.pct_change().dropna()

# Compute momentum and contrarian signals
momentum = returns.rolling(window=MOMENTUM_PERIOD).mean()
contrarian = returns.rolling(window=CONTRARIA_PERIOD).mean()

# Normalize signals
momentum = (momentum - momentum.mean()) / momentum.std()
contrarian = (contrarian - contrarian.mean()) / contrarian.std()


## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Generate signals
momentum_signals = momentum.shift(1)
contrarian_signals = contrarian.shift(1)

# Position sizing
momentum_weights = momentum_signals.rank(axis=1, pct=True)
contrarian_weights = contrarian_signals.rank(axis=1, pct=True)

# Portfolio construction
portfolio_weights = (momentum_weights + contrarian_weights) / 2
portfolio_weights = portfolio_weights.fillna(0)


## Phase 4 — Backtesting

In [ ]:
# Vectorized backtest
portfolio_returns = (returns * portfolio_weights).sum(axis=1)
cumulative_returns = (1 + portfolio_returns).cumprod()


## Phase 5 — Performance Metrics

In [ ]:
import scipy.stats as stats

# Performance metrics
annual_return = portfolio_returns.mean() * 12
annual_volatility = portfolio_returns.std() * np.sqrt(12)
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_volatility
sortino_ratio = (annual_return - RISK_FREE_RATE) / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(12)
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar_ratio = annual_return / (-max_drawdown)

print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

import matplotlib.pyplot as plt

# Plot equity curve
plt.figure(figsize=(10, 5))
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()


## Phase 6 — Monitoring

In [ ]:
# Monitoring stub
def monitor_portfolio(prices, weights):
    current_prices = prices.iloc[-1]
    current_positions = current_prices * weights.iloc[-1]
    daily_pnl = (current_prices - prices.iloc[-2]) * weights.iloc[-2]
    print(f'Daily P&L: {daily_pnl.sum():.2f}')
    print('Current Positions:')
    print(current_positions[current_positions!= 0])

# Example usage
monitor_portfolio(prices, portfolio_weights)
